## Imports & Setup

In [ ]:
!pip install optuna

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import random
import pickle
from abc import ABC, abstractmethod
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras import layers, regularizers, Input, Model, optimizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import seaborn as sns
import tensorflow as tf
import optuna

### Set random seeds

In [ ]:
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
tf.keras.utils.set_random_seed(42)

## Abstract Base Class

In [ ]:
class BaseForecastModel(ABC):
    """
    Simple base class for forecasting models.
    """

    def __init__(self, task_type: str, **hyperparameters):
        self.task_type = task_type
        self.hyperparameters = hyperparameters

    @abstractmethod
    def fit(self, X_train, y_train):
        pass

    @abstractmethod
    def predict(self, X):
        pass

    @abstractmethod
    def evaluate(self, X_test, y_test):
        pass

    @abstractmethod
    def save(self, filepath: str):
        pass

    @abstractmethod
    def load(self, filepath: str):
        pass

## Data Loading & Preprocessing

### Historical wheat futures prices data (1999-2025) from Investing.com

In [ ]:
df = pd.read_csv('US Wheat Futures Historical Data.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index('Date', drop=True)
df.sort_index(inplace=True)
df['Price'] = df['Price'].str.replace(',', '').astype('float64')
df_price = df['Price']
df_price

### Macroeconomics data (1999-08 ~ 2024-12) from FRED

In [ ]:
df_features = pd.read_csv('FRED.csv')
df_features = df_features.dropna(how='all', axis=1)
features_missing = []

for col in df_features.columns:
    if col == "Month":
        continue
    s = df_features[col].isna() | (df_features[col] == "")
    if not s.any():
        continue

    features_missing.append(col)

df_features['Month'] = pd.to_datetime(df_features['Month'])
df_features = df_features.set_index('Month').sort_index()
df_selected = df_features.drop(columns=features_missing)
features = [
    "RPI", "W875RX1", "CMRMTSPLx",
    "IPFPNSS", "USWTRADE",
    "USTRADE", "BUSLOANS", "CONSPI", "S&P 500",
    "S&P PE ratio", "FEDFUNDS", "TB3MS", "TB6MS", "GS1", "GS5",
    "GS10", "AAA", "BAA", "TB3SMFFM", "TB6SMFFM", "T1YFFM",
    "T5YFFM", "T10YFFM", "AAAFFM", "BAAFFM",
    "EXSZUSx", "EXJPUSx", "EXUSUKx", "EXCAUSx", "OILPRICEx",
    "PPICMM", "UMCSENTx"
]

df_features = df_features[features]
df_features

### Shift macro data by 1 month (release lag) and forward-fill to daily

In [ ]:
df_features = df_features.shift(periods=1, freq='infer').dropna()

start = df_features.index.min().replace(day=1)
end = (df_features.index.max() + pd.offsets.MonthEnd(0))
all_days = pd.date_range(start, end, freq='D')

df_features = df_features.reindex(all_days).ffill()
df_features.to_csv('features.csv')
df_features

### Inner join wheat futures + macro data, filter post-2008

In [ ]:
df_all = df_features.join(df_price, how='inner')
df_all = df_all['2008-01-01':]
df_all

### Build sliding-window dataset (lookback=30, one-step-ahead forecast)

In [ ]:
lookback = 30

data = df_all.values
X, y = [], []
for i in range(len(data) - lookback):
    seq_x = data[i : i + lookback, :]
    target = data[i + lookback, -1]
    X.append(seq_x)
    y.append(target)

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

### Price plot

In [ ]:
df_all['Price'].plot(figsize=(10, 5))

### 80/20 train-test split (no shuffling)

In [ ]:
test_size = int(X.shape[0] * 0.2)

X_train = X[:-test_size]
y_train = y[:-test_size]

X_test = X[-test_size:]
y_test = y[-test_size:]

print("Training set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)

## BiRNN with Skip Connection Model (implements BaseForecastModel)

Architecture:
- **Encoder**: Bidirectional SimpleRNN (returns full sequence)
- **Skip connection**: residual add around a second Bidirectional SimpleRNN block
- **Decoder**: SimpleRNN (returns last hidden state)
- **Output head**: Dense(1, linear)

In [ ]:
class BiRNNSkipForecastModel(BaseForecastModel):
    """
    Bidirectional SimpleRNN with skip/residual connection for
    one-step-ahead wheat futures price forecasting.

    Architecture:
        Input
          -> Bidirectional(SimpleRNN) [encoder, return_sequences=True]
          -> residual branch:
               copy ──────────────────────┐
               Bidirectional(SimpleRNN)    │  (skip connection)
               Add  <──────────────────────┘
          -> SimpleRNN [decoder, return last hidden state]
          -> Dense(1)
    """

    def __init__(
        self,
        task_type: str = 'regression',
        encoder_units: int = 64,
        skip_units: int = 64,
        decoder_units: int = 32,
        weight_decay: float = 0.02,
        dropout: float = 0.2,
        early_stop_patience: int = 10,
        epochs: int = 100,
        batch_size: int = 64,
        learning_rate: float = 1e-3,
    ):
        super().__init__(
            task_type=task_type,
            encoder_units=encoder_units,
            skip_units=skip_units,
            decoder_units=decoder_units,
            weight_decay=weight_decay,
            dropout=dropout,
            early_stop_patience=early_stop_patience,
            epochs=epochs,
            batch_size=batch_size,
            learning_rate=learning_rate,
        )

        self.encoder_units = encoder_units
        self.skip_units = skip_units
        self.decoder_units = decoder_units
        self.weight_decay = weight_decay
        self.dropout = dropout
        self.early_stop_patience = early_stop_patience
        self.epochs = epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate

        # Set during fit()
        self.model = None
        self.x_scaler = None
        self.y_scaler = None
        self.lookback = None
        self.n_features = None

    # ------------------------------------------------------------------ #
    #  Private helpers
    # ------------------------------------------------------------------ #

    def _build_model(self, seq_length: int, feature_dim: int) -> Model:
        """
        Construct the BiRNN + skip connection Keras model.

        The encoder Bidirectional(SimpleRNN) outputs 2*encoder_units per
        timestep. The skip block uses skip_units chosen so that
        2*skip_units == 2*encoder_units (i.e. skip_units == encoder_units)
        to allow the residual Add. If you want different encoder/skip
        widths, a Dense projection is inserted automatically.
        """
        inputs = Input(shape=(seq_length, feature_dim))

        # ---------- Encoder ---------- #
        x = layers.Bidirectional(
            layers.SimpleRNN(
                units=self.encoder_units,
                dropout=self.dropout,
                kernel_regularizer=regularizers.l2(self.weight_decay),
                return_sequences=True,
            )
        )(inputs)  # shape: (batch, seq_length, 2*encoder_units)

        # ---------- Skip / Residual block ---------- #
        residual = x  # save for skip

        x = layers.Bidirectional(
            layers.SimpleRNN(
                units=self.skip_units,
                dropout=self.dropout,
                kernel_regularizer=regularizers.l2(self.weight_decay),
                return_sequences=True,
            )
        )(x)  # shape: (batch, seq_length, 2*skip_units)

        # Project residual if widths don't match
        encoder_width = 2 * self.encoder_units
        skip_width = 2 * self.skip_units
        if encoder_width != skip_width:
            residual = layers.Dense(skip_width)(residual)

        x = layers.Add()([residual, x])  # skip connection

        # ---------- Decoder ---------- #
        x = layers.SimpleRNN(
            units=self.decoder_units,
            dropout=self.dropout,
            kernel_regularizer=regularizers.l2(self.weight_decay),
        )(x)  # returns last hidden state only

        # ---------- Output head ---------- #
        outputs = layers.Dense(
            units=1,
            activation='linear',
            kernel_regularizer=regularizers.l2(self.weight_decay),
        )(x)

        return Model(inputs=inputs, outputs=outputs)

    def _scale_X(self, X, fit: bool = False):
        """Flatten -> scale -> reshape back to 3-D."""
        n, lb, nf = X.shape
        X_flat = X.reshape(n, lb * nf)
        if fit:
            self.x_scaler = StandardScaler()
            X_flat = self.x_scaler.fit_transform(X_flat)
        else:
            X_flat = self.x_scaler.transform(X_flat)
        return X_flat.reshape(n, lb, nf)

    # ------------------------------------------------------------------ #
    #  Required interface methods
    # ------------------------------------------------------------------ #

    def fit(self, X_train, y_train):
        """
        Train on the last fold of 5-fold expanding-window CV.

        Features and targets are both standardized per fold.
        Uses early stopping and LR reduction on plateau.
        """
        self.lookback = X_train.shape[1]
        self.n_features = X_train.shape[2]

        tscv = TimeSeriesSplit(n_splits=5)

        for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
            if fold < 4:
                continue

            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]

            X_tr_s = self._scale_X(X_tr, fit=True)
            X_val_s = self._scale_X(X_val, fit=False)

            self.y_scaler = StandardScaler()
            y_tr_s = self.y_scaler.fit_transform(y_tr.reshape(-1, 1))
            y_val_s = self.y_scaler.transform(y_val.reshape(-1, 1))

            self.model = self._build_model(self.lookback, self.n_features)
            self.model.compile(
                optimizer=Adam(learning_rate=self.learning_rate),
                loss='mean_squared_error',
            )

            callbacks = [
                EarlyStopping(
                    monitor='val_loss',
                    patience=self.early_stop_patience,
                    restore_best_weights=True,
                    verbose=0,
                ),
                ReduceLROnPlateau(
                    monitor='val_loss',
                    factor=0.5,
                    patience=5,
                    min_lr=1e-6,
                    verbose=0,
                ),
            ]

            self.model.fit(
                X_tr_s, y_tr_s,
                validation_data=(X_val_s, y_val_s),
                epochs=self.epochs,
                batch_size=self.batch_size,
                shuffle=False,
                callbacks=callbacks,
                verbose=0,
            )

    def predict(self, X):
        """
        Predict wheat futures prices.

        Returns:
            predictions: np.ndarray of shape (n_samples,)
        """
        if self.model is None:
            raise ValueError("Model not trained! Call fit() first.")

        X_s = self._scale_X(X, fit=False)
        y_pred_scaled = self.model.predict(X_s, verbose=0)
        y_pred = self.y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1))
        return y_pred.flatten()

    def evaluate(self, X_test, y_test):
        """
        Evaluate model on test data.

        Returns:
            dict with 'rmse', 'mae', and 'r2'
        """
        predictions = self.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        mae = mean_absolute_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)

        return {'rmse': rmse, 'mae': mae, 'r2': r2}

    def save(self, filepath: str):
        """Save model weights, scalers, and hyperparameters."""
        self.model.save_weights(filepath + '.weights.h5')

        meta = {
            'hyperparameters': self.hyperparameters,
            'x_scaler': self.x_scaler,
            'y_scaler': self.y_scaler,
            'lookback': self.lookback,
            'n_features': self.n_features,
            'encoder_units': self.encoder_units,
            'skip_units': self.skip_units,
            'decoder_units': self.decoder_units,
        }
        with open(filepath + '.meta.pkl', 'wb') as f:
            pickle.dump(meta, f)

        print(f"Model saved to {filepath}.*")

    def load(self, filepath: str):
        """Load a previously saved model."""
        with open(filepath + '.meta.pkl', 'rb') as f:
            meta = pickle.load(f)

        self.x_scaler = meta['x_scaler']
        self.y_scaler = meta['y_scaler']
        self.lookback = meta['lookback']
        self.n_features = meta['n_features']
        self.encoder_units = meta['encoder_units']
        self.skip_units = meta['skip_units']
        self.decoder_units = meta['decoder_units']

        self.model = self._build_model(self.lookback, self.n_features)
        self.model.load_weights(filepath + '.weights.h5')

        print(f"Model loaded from {filepath}.*")

## Optuna Hyperparameter Search (BiRNN + Skip Connection)

In [ ]:
def cv_birnn_skip(X_train, y_train, **kwargs):
    """Run 5-fold expanding-window CV and return mean val loss."""
    tscv = TimeSeriesSplit(n_splits=5)
    lookback, n_features = X_train.shape[1], X_train.shape[2]
    val_losses = []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        ns_tr, ns_val = X_tr.shape[0], X_val.shape[0]
        X_tr_flat = X_tr.reshape((ns_tr, lookback * n_features))
        X_val_flat = X_val.reshape((ns_val, lookback * n_features))

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr_flat).reshape((ns_tr, lookback, n_features))
        X_val_s = scaler.transform(X_val_flat).reshape((ns_val, lookback, n_features))

        y_scaler = StandardScaler()
        y_tr_s = y_scaler.fit_transform(y_tr.reshape(-1, 1))
        y_val_s = y_scaler.transform(y_val.reshape(-1, 1))

        tmp = BiRNNSkipForecastModel(**kwargs)
        model = tmp._build_model(lookback, n_features)
        model.compile(
            optimizer=Adam(learning_rate=kwargs.get('learning_rate', 1e-3)),
            loss='mean_squared_error',
        )

        early_stop = EarlyStopping(
            monitor='val_loss',
            patience=kwargs.get('early_stop_patience', 10),
            verbose=0,
            restore_best_weights=True,
        )
        reduce_lr = ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0,
        )

        history = model.fit(
            X_tr_s, y_tr_s,
            validation_data=(X_val_s, y_val_s),
            epochs=kwargs.get('epochs', 100),
            batch_size=kwargs.get('batch_size', 64),
            shuffle=False,
            callbacks=[early_stop, reduce_lr],
            verbose=0,
        )
        val_losses.append(min(history.history['val_loss']))

    return np.mean(val_losses)

In [ ]:
def objective(trial):
    # encoder_units and skip_units are kept equal so the
    # residual Add works without a projection layer
    shared_units = trial.suggest_categorical('encoder_units', [32, 64, 128])

    params = dict(
        encoder_units=shared_units,
        skip_units=shared_units,
        decoder_units=trial.suggest_categorical('decoder_units', [32, 64, 128]),
        weight_decay=trial.suggest_float('weight_decay', 5e-4, 2e-2, log=True),
        dropout=trial.suggest_float('dropout', 0.1, 0.3),
        early_stop_patience=trial.suggest_int('early_stop_patience', 10, 15),
        epochs=trial.suggest_int('epochs', 50, 200),
        batch_size=trial.suggest_categorical('batch_size', [32, 64]),
        learning_rate=trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True),
    )
    return cv_birnn_skip(X_train, y_train, **params)


study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

print('Best CV score (val_loss):', study.best_value)
print('Best hyperparameters:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

## Train, Evaluate & Plot (BiRNN + Skip with best hyperparameters)

In [ ]:
# ---- Instantiate with optimized hyperparameters ---- #
# Replace these with study.best_params after running Optuna above.
# Example placeholder values shown below:
best_params = study.best_params

best_model = BiRNNSkipForecastModel(
    task_type='regression',
    encoder_units=best_params['encoder_units'],
    skip_units=best_params['encoder_units'],  # same as encoder for clean residual
    decoder_units=best_params['decoder_units'],
    weight_decay=best_params['weight_decay'],
    dropout=best_params['dropout'],
    early_stop_patience=best_params['early_stop_patience'],
    epochs=best_params['epochs'],
    batch_size=best_params['batch_size'],
    learning_rate=best_params['learning_rate'],
)

# ---- fit ---- #
best_model.fit(X_train, y_train)

# ---- evaluate ---- #
test_metrics = best_model.evaluate(X_test, y_test)
print('\n=== Test set performance (BiRNN + Skip) ===')
for name, value in test_metrics.items():
    print(f'  {name}: {value:.6f}')

# ---- plot ---- #
test_preds = best_model.predict(X_test)
test_dates = df_all.index[-len(X_test):]

plt.figure(figsize=(12, 5))
plt.plot(test_dates, y_test, label='Actual (test)')
plt.plot(test_dates, test_preds, label='Predicted (test)')
plt.title('BiRNN + Skip Connection: Actual vs Predicted (Test)')
plt.xlabel('Date')
plt.ylabel('Price')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## Save & Load

In [ ]:
# Save
best_model.save('birnn_skip_wheat')

# Load into a fresh instance and verify
loaded = BiRNNSkipForecastModel()
loaded.load('birnn_skip_wheat')

loaded_metrics = loaded.evaluate(X_test, y_test)
print('\nLoaded model test metrics:')
for name, value in loaded_metrics.items():
    print(f'  {name}: {value:.6f}')

print(f"\nPredictions match: {np.allclose(best_model.predict(X_test), loaded.predict(X_test))}")

## Naive Baseline

In [ ]:
last_train_price = y_train[-1]
naive_preds = np.empty(len(y_test))
naive_preds[0] = last_train_price
naive_preds[1:] = y_test[:-1]

naive_rmse = np.sqrt(mean_squared_error(y_test, naive_preds))
naive_mae = mean_absolute_error(y_test, naive_preds)
naive_r2 = r2_score(y_test, naive_preds)

print('=== Naive One-Day-Ahead Forecast ===')
print(f'RMSE: {naive_rmse:.6f}')
print(f'R²  : {naive_r2:.6f}')
print(f'MAE : {naive_mae:.6f}')